In [6]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from CONFIG import GROQ_MODEL
from dotenv import load_dotenv
from langchain_core.tools import tool
import re
from typing import Any
from langchain.agents.middleware import PIIMiddleware
from langchain.agents import create_agent

In [7]:
load_dotenv()
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

In [8]:
@tool
def look_up(query: str) -> str:
    "look up customer information"
    return f"customer record found for query {query}"

In [25]:
agent = create_agent(
    model=llm,
    tools=[look_up],
    middleware=[
        PIIMiddleware(
            'email',
            strategy='redact',
            apply_to_input=True
	  ),
	  PIIMiddleware(
		'credit_card',
		strategy='mask',
		apply_to_input=True
	  ),
	  PIIMiddleware(
		'api_key',
		detector=r"sk-[a-zA-Z0-9]{32}",
		strategy="block",
		apply_to_input=True
	  )
    ]
)

In [22]:
user_input = "my email is saeedadnan151@gmail.com, my credit card number is 5420-1181-2979-5090 can you help me?"

respo = agent.invoke(
    {
        'messages': [
            {
                'role': 'user',
                'content': user_input
		}
	  ]
    }
)

In [23]:
print(respo['messages'][-1].content)

I cannot assist with requests involving sensitive personal or financial information. Is there anything else I can help you with?


In [24]:
respo

{'messages': [HumanMessage(content='my email is [REDACTED_EMAIL], my credit card number is 5420-1181-2979-5090 can you help me?', additional_kwargs={}, response_metadata={}, id='d3dbb577-c71c-4cf2-a1de-3495de2ae5fe'),
  AIMessage(content='I cannot assist with requests involving sensitive personal or financial information. Is there anything else I can help you with?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 240, 'total_tokens': 263, 'completion_time': 0.104477947, 'completion_tokens_details': None, 'prompt_time': 0.012169553, 'prompt_tokens_details': None, 'queue_time': 0.008558567, 'total_time': 0.1166475}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cc7cc-3ecf-7311-bcd1-1529a8edfec0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 240, 'output_tokens': 23

In [30]:
user_input = "user api key is sk-k8Fz2Qm7Xr4Lp9Ta1Wc6Vb3Nd0Hy5JuE can you help me?"

try:
	respo = agent.invoke(
	{
		'messages': [
			{
			'role': 'user',
			'content': user_input
			}
		]
	}
	)
	print(respo)
except Exception as e:
	print(f"BLOCKED: {e}")

BLOCKED: Detected 1 instance(s) of api_key in text content
